# Root Material Map Analysis
This is for material maps produced for geometry Run4-05-00-00 with my local changes to ACTS for the additional element and fraction information. There is no offical ACTS group released map to compare to yet <br>
<br>
The tracking geometry description is a different generation from that which that JSON material maps were produced with. Therefore the ROOT and JSON material maps are not to be compared, this is analysis is just to showcase the lack of empty bins and the reasonable magnitude of the thickness values. <br>
<br>
A detailed analysis of the ROOT material map was not necessary as the maps produced from ACTS v46.0 and onwards were easily validated. This is notebook is intentionally left open-ended for any ROOT material map analysis that one would like to perform. 

In [1]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Any, Dict, Iterable, Tuple, Set
from mpl_toolkits.mplot3d import Axes3D
import itertools
import collections
import awkward as ak
import uproot
from collections import Counter
from collections import defaultdict
import matplotlib.colors as mcolors

In [2]:
f = uproot.open("/home/lwills/masters-thesis/Files/material-maps/DRAFT_Run4-05-00-material-maps.root")

In [3]:
import numpy as np

def get_surface_info(f, surface_list):
    results = []
    for i, (vol, bou, lay, app, sen) in enumerate(surface_list, start=1):
        key = f"SurfaceMaterial_vol{vol}_bou{bou}_lay{lay}_app{app}_sen{sen}"
        try:
            name = f[f"{key}/v"].to_numpy()[0]
            num = f[f"{key}/n"].to_numpy()[0]
            mn = f[f"{key}/min"].to_numpy()[0]
            mx = f[f"{key}/max"].to_numpy()[0]
            bou_val = f[f"{key}/o"].to_numpy()[0]
        except KeyError as e:
            print(f"[sur{i}] key not found: {key} ({e})")
            continue

        info = {
            "surface": (vol, bou, lay, app, sen),
            "key": key,
            "name": name,
            "num": num,
            "min": mn,
            "max": mx,
            "bou": bou_val,
        }
        results.append(info)
        print(f"[sur{i}] vol{vol} bou{bou} lay{lay} app{app} sen{sen}: bin names: {name} binning : {num}, min: {mn}, max: {mx}")

    return results

In [4]:
surface_list = [(4, 1, 1, 0, 0), (5, 3, 1, 0, 0), (6, 1, 1, 0, 0), (9, 1, 1, 0, 0), (10, 1, 2, 0, 0), (11, 1, 3, 0, 0), (19, 1, 1, 0, 0), (20, 1, 2, 0, 0)]
results = get_surface_info(f, surface_list)

[sur1] vol4 bou1 lay1 app0 sen0: bin names: [3. 4.] binning : [50. 50.], min: [23.934     -3.1415927], max: [123.7396      3.1415927]
[sur2] vol5 bou3 lay1 app0 sen0: bin names: [5. 2.] binning : [ 50. 100.], min: [-124.29352 -254.9875 ], max: [124.29352 254.9875 ]
[sur3] vol6 bou1 lay1 app0 sen0: bin names: [3. 4.] binning : [50. 50.], min: [23.934     -3.1415927], max: [123.7396      3.1415927]
[sur4] vol9 bou1 lay1 app0 sen0: bin names: [3. 4.] binning : [50. 50.], min: [123.7396     -3.1415927], max: [205.10211     3.1415927]
[sur5] vol10 bou1 lay2 app0 sen0: bin names: [5. 2.] binning : [ 50. 100.], min: [-739.82196 -380.73083], max: [739.82196 380.73083]
[sur6] vol11 bou1 lay3 app0 sen0: bin names: [3. 4.] binning : [50. 50.], min: [123.7396     -3.1415927], max: [205.10211     3.1415927]
[sur7] vol19 bou1 lay1 app0 sen0: bin names: [3. 4.] binning : [50. 50.], min: [316.72556    -3.1415927], max: [1015.9019       3.1415927]
[sur8] vol20 bou1 lay2 app0 sen0: bin names: [5. 2.] bi

In [5]:
tree = f['IndexedMaterial']
print(tree.keys())
#ElementZ and ElementFrac are the additional material vectors

['t', 'x0', 'l0', 'A', 'Z', 'rho', 'ElementZ', 'ElementFrac']


In [6]:
surfaces = [k.rstrip(';1') for k in f.keys() 
            if '/' not in k and 'IndexedMaterial' not in k]

tree = f["IndexedMaterial"]
t_arr = tree["t"].array(library="np")
x0_arr = tree["x0"].array(library="np")
l0_arr = tree["l0"].array(library="np")
A_arr = tree["A"].array(library="np")
Z_arr = tree["Z"].array(library="np")
rho_arr  = tree["rho"].array(library="np")
elemZ_arr = tree["ElementZ"].array(library="ak")
elemF_arr = tree["ElementFrac"].array(library="ak")

material_map = {}

for surf in surfaces:
    parts = surf.replace('SurfaceMaterial_', '').split('_')
    volume = int(parts[0].replace('vol', ''))
    boundary = int(parts[1].replace('bou', ''))
    layer = int(parts[2].replace('lay', ''))
    approach = int(parts[3].replace('app', ''))
    sur_id = (int(volume), int(boundary), int(layer), int(approach))
    n = f[f"{surf}/n"].to_numpy()[0]
    v = f[f"{surf}/v"].to_numpy()[0]
    mins = f[f"{surf}/min"].to_numpy()[0]
    maxs = f[f"{surf}/max"].to_numpy()[0]
    idx_grid = f[f"{surf}/i"].to_numpy()[0]
    max_valid_idx = min(len(t_arr), 
                    len(x0_arr), len(l0_arr), len(A_arr), 
                    len(Z_arr), len(rho_arr)) - 1

    n0, n1 = int(n[0]), int(n[1])
    idx_grid = idx_grid.reshape(n0, n1)

    edges_0 = np.linspace(mins[0], maxs[0], n0 + 1)
    edges_1 = np.linspace(mins[1], maxs[1], n1 + 1)
    axis_name_0 = str(v[0])
    axis_name_1 = str(v[1])
    material_map[surf] = {}
    for i0 in range(n0):
        for i1 in range(n1):
            mat_idx = idx_grid[i0, i1]
            bin_key = (i0, i1)
            center_0 = 0.5 * (edges_0[i0] + edges_0[i0 + 1])
            center_1 = 0.5 * (edges_1[i1] + edges_1[i1 + 1])
            material_map[surf][bin_key] = {
                "surface": sur_id,
                "bin_key": bin_key,
                "center0": float(center_0),
                "center1": float(center_1),
                "mat_idx": int(mat_idx),
                "t": float(t_arr[mat_idx]),
                "x0": float(x0_arr[mat_idx]),
                "l0": float(l0_arr[mat_idx]),
                "A": float(A_arr[mat_idx]),
                "Z": float(Z_arr[mat_idx]),
                "rho": float(rho_arr[mat_idx]),
                "ElementZ": [int(z)    for z in elemZ_arr[mat_idx]],
                "ElementFrac":[float(ff) for ff in elemF_arr[mat_idx]],
            }
print(f"Surfaces loaded: {len(material_map)}")

Surfaces loaded: 202


In [19]:
total_bins = sum(len(bins) for bins in material_map.values())
print(f"Total bins: {total_bins}")

Total bins: 535035
535035

In [40]:
IS_negEC_volume = []
for sur, bins in material_map.items():
    for e in bins.values():
        if (e['surface'][0] == 4) and (e['surface'][1] == 1) and (e['surface'][2] == 1):
            bin_key = e['bin_key']
            if bin_key[0] == 1 and bin_key[1] == 30:
                print(f"Example Bin from Negative End-Cap Surface: {e}")


Example Bin from Negative End-Cap Surface: {'surface': (4, 1, 1, 0), 'bin_key': (1, 30), 'center0': 26.92816925048828, 'center1': 0.691150426864624, 'mat_idx': 6536, 't': 26.14776611328125, 'x0': 251.7557373046875, 'l0': 443.6619567871094, 'A': 11.394009590148926, 'Z': 5.835074424743652, 'rho_gcm3': 8.996577796494815e+20, 'ElementZ': [1, 6, 7, 8, 13, 18], 'ElementFrac': [0.11857247352600098, 0.6999049186706543, 0.04921620339155197, 0.07041294872760773, 0.0614180713891983, 0.0004753758548758924]}


In [50]:
#Just a break down of the average amount of each material present in the material map
element_fractions = defaultdict(list)
for sur, bins in material_map.items():
    for b in bins.values():
        for z, frac in zip(b['ElementZ'], b['ElementFrac']):
            element_fractions[z].append(frac)
for z in sorted(element_fractions.keys()):
    fracs = element_fractions[z]
    print(f"Z={z}: mean= {np.mean(fracs):.4f}, std = {np.std(fracs):.4f}")

Z=1: mean= 0.0389, std = 0.0522
Z=4: mean= 0.0028, std = 0.0037
Z=5: mean= 0.0226, std = 0.0133
Z=6: mean= 0.6474, std = 0.2467
Z=7: mean= 0.0167, std = 0.0290
Z=8: mean= 0.0555, std = 0.0642
Z=11: mean= 0.0005, std = 0.0005
Z=12: mean= 0.0009, std = 0.0010
Z=13: mean= 0.0545, std = 0.1372
Z=14: mean= 0.0670, std = 0.0798
Z=15: mean= 0.0001, std = 0.0001
Z=16: mean= 0.0001, std = 0.0002
Z=17: mean= 0.0178, std = 0.0173
Z=18: mean= 0.0013, std = 0.0027
Z=22: mean= 0.0691, std = 0.1124
Z=23: mean= 0.0008, std = 0.0015
Z=24: mean= 0.0054, std = 0.0104
Z=25: mean= 0.0007, std = 0.0013
Z=26: mean= 0.0247, std = 0.0769
Z=27: mean= 0.0003, std = 0.0020
Z=28: mean= 0.0045, std = 0.0136
Z=29: mean= 0.1098, std = 0.1388
Z=30: mean= 0.0057, std = 0.0049
Z=35: mean= 0.0261, std = 0.0172
Z=40: mean= 0.0000, std = 0.0000
Z=42: mean= 0.0003, std = 0.0003
Z=47: mean= 0.0005, std = 0.0005
Z=50: mean= 0.0132, std = 0.0134
Z=56: mean= 0.0062, std = 0.0060
Z=79: mean= 0.0000, std = 0.0001
Z=82: mean= 0.00

## Is the Thickness a reasonable value:
Are the thickness values assigned to representative surfaces (excluding surfaces under volumes 1, 2, and 3) a reasonable value, such that their converted volumes will fit in the $\approx 6 \mathrm{m}$ ITk detector:

In [55]:
thick_per_volume = defaultdict(list)
for sur, bins in material_map.items():
    for b in bins.values():
        volume = b['surface'][0]
        layer = b['surface'][2]
        bin0 = b['bin_key'][0]
        binkey = b['bin_key']
        thick_per_volume[volume].append(b['t'])
for volume, thickness in sorted(thick_per_volume.items()):
    print(f" volume: {volume} thickness: {np.mean(thickness):.4f} std: {np.std(thickness):.4f}")

 volume: 1 thickness: 25.9902 std: 1.1852
 volume: 2 thickness: 452.6079 std: 1462.1099
 volume: 3 thickness: 452.4339 std: 1467.4362
 volume: 4 thickness: 5.2947 std: 17.5393
 volume: 5 thickness: 1.6494 std: 2.3065
 volume: 6 thickness: 5.2655 std: 17.1607
 volume: 7 thickness: 61.4227 std: 17.6492
 volume: 9 thickness: 2.5677 std: 3.8919
 volume: 10 thickness: 2.0475 std: 1.6356
 volume: 11 thickness: 2.5914 std: 3.9377
 volume: 19 thickness: 14.9195 std: 15.7250
 volume: 20 thickness: 10.2214 std: 4.4850
 volume: 21 thickness: 14.8543 std: 15.6512
 volume: 30 thickness: 28.2407 std: 27.8518
 volume: 31 thickness: 28.2288 std: 27.8881


In [56]:
results = []

for sur, bins in material_map.items():
    thick_per_surf = []
    for b in bins.values():
        thick_per_surf.append(b['t'])
    
    results.append({
        'surface':  sur,
        'avg_thickness': np.mean(thick_per_surf),
        'std_thickness': np.std(thick_per_surf),
        'n_bins': len(thick_per_surf)
    })
averages = []
#for r in results:
#    print(f"Surface: {r['surface']},  avg = {r['avg_thickness']:.4f},  std = {r['std_thickness']:.4f},  n = {r['n_bins']}")
#    averages.append(r['avg_thickness'])
#uncomment the loop above for a per-surface study of thickness   